# Track B on Kaggle — runs while your browser is closed

**One-time:** Kaggle account with a verified phone number (needed for *Internet* and GPU).

**Each time**
1. Kaggle → *Create → New Notebook* → *File → Import Notebook* → upload this file.
2. Right panel → *Session options*: **Accelerator = GPU T4 x2**, **Internet = On**.
3. Edit cell 1 (which matches, period times, direction).
4. Top right **Save Version → Save & Run All (Commit) → Save**. You can close the browser now.
5. Later: open the notebook → *Output* (of that version) → download `deliverables/*.zip` and the smoke stills.

Two matches run at the same time (one per T4). A run may last up to 12 hours; a 2-match batch should take roughly 1.5–2 h (estimate). The GPU hours count against your weekly Kaggle quota (shown in your Kaggle settings).

**Resume after a failed run:** *Add Input* → *Notebook Output* → pick the previous version of this notebook, then commit again; finished chunks are copied in and skipped.

**If YouTube refuses the download** (bot check): upload the mp4 as a Kaggle Dataset (file named `<match_id>.mp4` or `<youtube-id>.mp4`), add it via *Add Input*, and commit again.

In [ ]:
# 1 · What to run — edit, then Save & Run All (Commit)
# Batch for the Pekan 5 game at Persibangga (30 Nov): their 4 matches first, then PERSIBA vs Persinga.
# 2 run at once (one per T4); about 5-6 h for all five. Order = priority.
MATCHES = [
    "gresik_persibangga_20261126",    # Pekan 4  Gresik 2-1 Persibangga
    "persiba_persinga_20261126",      # Pekan 4  PERSIBA 0-1 Persinga
    "persinga_persibangga_20261122",  # Pekan 3  Persinga 1-1 Persibangga
    "persibangga_persibo_20261118",   # Pekan 2  Persibangga 2-1 Persibo
    "psf_persibangga_20261114",       # Pekan 1  PSF 2-0 Persibangga
]
# Next batch (after this one): "persiba_gresik_20261113", "persipa_persiba_20261117"

# Everything above already has its period times and video in the config. The attack direction
# is read from the data. Only fill this if something must be overridden, e.g.
OVERRIDES = {
    # "persiba_persinga_20261126": {"swap_teams": True},   # only if the smoke stills show the HOME team in ORANGE
}
BRANCH = "main"
READ_NUMBERS = True    # read shirt numbers with OCR (a bit slower; pre-fills player numbers in review)

In [ ]:
# 2 · Setup (about 3 min)
import os, sys, json, glob, shutil, subprocess
from pathlib import Path

REPO, ROOT = Path("/kaggle/working/football-analytics"), Path("/kaggle/working/fa")
WORK, VIDEOS = Path("/tmp/work"), Path("/tmp/videos")   # not saved: keeps the output small
for d in (ROOT, WORK, VIDEOS):
    d.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                "https://github.com/mryudistira21/football-analytics.git", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt"),
                "git+https://github.com/roboflow/sports.git", "psutil",
                *(["easyocr"] if READ_NUMBERS else [])], check=True)
if shutil.which("deno") is None and shutil.which("node") is None:
    subprocess.run("curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1", shell=True)
    os.environ["PATH"] = str(Path.home() / ".deno" / "bin") + os.pathsep + os.environ["PATH"]

# Config = repo config + your overrides (kept in the output for reference)
cfg = json.loads((REPO / "matches" / "persiba_2026-27.json").read_text())
by_id = {m["match_id"]: m for m in cfg["matches"]}
for mid, ov in OVERRIDES.items():
    if mid not in by_id:
        raise SystemExit(f"Unknown match_id in OVERRIDES: {mid}")
    ov = dict(ov)
    if "periods" in ov:   # placeholders ("0:00", "0:00") keep whatever the config has
        merged = dict(by_id[mid].get("periods") or {})
        merged.update({k: v for k, v in ov["periods"].items() if v and list(v) != ["0:00", "0:00"]})
        ov["periods"] = merged
    by_id[mid].update({k: v for k, v in ov.items() if v is not None})
CFG = ROOT / "config.json"
CFG.write_text(json.dumps(cfg, indent=2))

# Resume: copy runs from an attached previous output; link uploaded videos
for prev in glob.glob("/kaggle/input/*/fa/runs/*") + glob.glob("/kaggle/input/*/*/fa/runs/*"):
    dst = ROOT / "runs" / Path(prev).name
    if not dst.exists():
        shutil.copytree(prev, dst)
        print("resumed from", prev)
for v in glob.glob("/kaggle/input/**/*.mp4", recursive=True):
    link = VIDEOS / Path(v).name
    if not link.exists():
        link.symlink_to(v)
        print("video from dataset:", v)

print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
GPUS = ",".join(str(i) for i in range(max(1, len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()))))
print("GPUs used:", GPUS)

In [ ]:
# 3 · Run (smoke test + full match per match, one per GPU). Progress prints every 30 s.
os.chdir(REPO)
cmd = [sys.executable, "scripts/run_queue.py", "--config", str(CFG), "--matches", ",".join(MATCHES),
       "--root", str(ROOT), "--work", str(WORK), "--video-dir", str(VIDEOS), "--gpus", GPUS, "--download", "--no-db", *(["--read-numbers"] if READ_NUMBERS else [])]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("exit code", proc.returncode)

In [ ]:
# 4 · Collect the deliverables (bundles + smoke stills + logs) in one folder
OUT = Path("/kaggle/working/deliverables"); OUT.mkdir(exist_ok=True)
for mid in MATCHES:
    run = ROOT / "runs" / mid
    for f in list(run.glob("*_trackb_bundle.zip")) + list((run / "smoke").glob("still_*.jpg")) + list((run / "smoke").glob("smoke_summary.json")):
        dst = OUT / (f.name if f.suffix == ".zip" else f"{mid}_{f.name}")
        shutil.copy(f, dst)
    log = ROOT / "queue" / f"{mid}.log"
    if log.exists():
        shutil.copy(log, OUT / f"{mid}.log")
shutil.copy(CFG, OUT / "config_used.json")
shutil.rmtree(REPO, ignore_errors=True)   # code is on GitHub; keep the output small
for p in sorted(OUT.iterdir()):
    print(f"{p.name:60s} {p.stat().st_size/2**20:8.1f} MB")
print(json.dumps(json.loads((ROOT / "queue" / "status.json").read_text()), indent=1))